# Importing

## Import Library

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score




# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## Import CSV And convert to DataFrame

In [ ]:
df = pd.read_csv('/kaggle/input/ecommerce-customer-behavior-dataset/ecommerce_customer_churn_dataset.csv')

# Preprocessing

## Frist five row

In [ ]:
df.head()

## last Five row

In [ ]:
df.tail()

## Shape of our dataset

In [ ]:
df.shape

## List out all columns

In [ ]:
df.columns

## Datatype of each columns

In [ ]:
df.dtypes

## Information of all Columns

In [ ]:
df.info()

## Check Null Value

In [ ]:
df.isnull().sum().sort_values(ascending=False)

## Drop High Count Null Values

In [ ]:
df.drop(columns=[
    'Social_Media_Engagement_Score',
    'Mobile_App_Usage',
    'Credit_Balance'
], inplace=True)

In [ ]:
df.isnull().sum()

## Handle Int and Float Value By Median

In [ ]:
num_cols = df.select_dtypes(include=['int64','float64']).columns

for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

## Handle Categorical Column by Replace With Mode

In [ ]:
cat_cols = df.select_dtypes(include='object').columns

for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

df.isnull().sum()

## Check Dupicate Value

In [ ]:
df.duplicated().sum()

## Summary

In [ ]:
df.describe()

# EDA

In [ ]:
def show_fig():
    plt.tight_layout()
    plt.show()

plot_no = 1

In [ ]:
fig = plt.figure(figsize=(10,6))
sns.histplot(df['Age'], bins=40, kde=True)
plt.title(f'{plot_no}. Age Distribution of Customers Showing Core Demographic Spread')
show_fig()
plot_no += 1


In [ ]:
fig = plt.figure(figsize=(10,6))
sns.countplot(x='Gender', hue='Churned', data=df)
plt.title(f'{plot_no}. Gender-wise Churn Behavior Comparison')
show_fig()
plot_no += 1


In [ ]:
fig = plt.figure(figsize=(10,6))
sns.barplot(x='Signup_Quarter', y='Lifetime_Value', hue='Churned', data=df)
plt.title(f'{plot_no}. Lifetime Value Trends Across Signup Quarters and Churn')
show_fig()
plot_no += 1


In [ ]:
fig = plt.figure(figsize=(10,6))
sns.histplot(df['Lifetime_Value'], bins=50, kde=True)
plt.title(f'{plot_no}. Distribution of Customer Lifetime Value')
show_fig()
plot_no += 1


In [ ]:
fig = plt.figure(figsize=(10,6))
sns.scatterplot(x='Login_Frequency', y='Session_Duration_Avg', hue='Churned', data=df, alpha=0.5)
plt.title(f'{plot_no}. Engagement Patterns Based on Login Frequency and Session Duration')
show_fig()
plot_no += 1


In [ ]:
fig = plt.figure(figsize=(10,6))
sns.scatterplot(x='Pages_Per_Session', y='Cart_Abandonment_Rate', hue='Churned', data=df, alpha=0.5)
plt.title(f'{plot_no}. Cart Abandonment Behavior vs Browsing Depth')
show_fig()
plot_no += 1


In [ ]:
fig = plt.figure(figsize=(10,6))
sns.boxplot(x='Churned', y='Cart_Abandonment_Rate', data=df)
plt.title(f'{plot_no}. Cart Abandonment Rate Difference Between Churned and Active Users')
show_fig()
plot_no += 1


In [ ]:
fig = plt.figure(figsize=(10,6))
sns.scatterplot(x='Wishlist_Items', y='Lifetime_Value', hue='Churned', data=df, alpha=0.5)
plt.title(f'{plot_no}. Relationship Between Wishlist Usage and Customer Value')
show_fig()
plot_no += 1


In [ ]:
fig = plt.figure(figsize=(10,6))
sns.barplot(x='Payment_Method_Diversity', y='Lifetime_Value', data=df)
plt.title(f'{plot_no}. Effect of Payment Method Diversity on Lifetime Value')
show_fig()
plot_no += 1


In [ ]:
fig = plt.figure(figsize=(10,6))
sns.scatterplot(x='Discount_Usage_Rate', y='Lifetime_Value', hue='Churned', data=df, alpha=0.5)
plt.title(f'{plot_no}. Discount Dependency and Its Impact on Customer Value')
show_fig()
plot_no += 1


In [ ]:
fig = plt.figure(figsize=(10,6))
sns.boxplot(x='Churned', y='Email_Open_Rate', data=df)
plt.title(f'{plot_no}. Email Engagement Comparison for Churned vs Retained Customers')
show_fig()
plot_no += 1


In [ ]:
fig = plt.figure(figsize=(10,6))
sns.barplot(x='Country', y='Churned', data=df)
plt.title(f'{plot_no}. Country-wise Average Churn Rate Comparison')
show_fig()
plot_no += 1


In [ ]:
fig = plt.figure(figsize=(10,6))
sns.scatterplot(x='Product_Reviews_Written', y='Lifetime_Value', hue='Churned', data=df, alpha=0.5)
plt.title(f'{plot_no}. Customer Advocacy Through Reviews and Value Contribution')
show_fig()
plot_no += 1


In [ ]:
fig = plt.figure(figsize=(10,6))
sns.histplot(df['Session_Duration_Avg'], bins=40, kde=True)
plt.title(f'{plot_no}. Distribution of Average Session Duration Indicating Engagement Health')
show_fig()
plot_no += 1


# Model Training

## Select Fetures and Target

In [ ]:
X = df.drop('Churned', axis=1)
y = df['Churned']

## Saperate Object and Numeric Features

In [ ]:
cat_cols = X.select_dtypes(include='object').columns
num_cols = X.select_dtypes(exclude='object').columns

## Transformer Column

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', MinMaxScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ]
)

## Split data in 80:20 Ratio

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## Create GradentBoostingClassifier Model

In [ ]:
model = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

## Create Pipeline

In [ ]:
pipeline = Pipeline(
    steps=[
        ('preprocessing', preprocessor),
        ('model', model)
    ]
)

## Train Pipeline

In [ ]:
pipeline.fit(X_train, y_train)

## give 20% Tested data to check Accureccy

In [ ]:
y_pred = pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred) * 100
accuracy

In [ ]:
# We beat 92% accureccy
# so we can say that our model accuracy is 92%

## Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix | Accuracy = {accuracy:.2f}%')
plt.tight_layout()
plt.show()